In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
from data import df

In [2]:
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# funkcja dzieląca tekst na mniejsze fragmenty o określonej liczbie tokenów

def split_into_chunks(text, tokenizer,
                      max_tokens=510,
                      overlap=50):

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    step = max_tokens - overlap

    for i in range(0, len(token_ids), step):

        chunk = token_ids[i:i + max_tokens]

        if len(chunk) == 0:
            break

        chunks.append(
            tokenizer.decode(
                chunk,
                skip_special_tokens=True
            )
        )

        if i + max_tokens >= len(token_ids):
            break

    return chunks

In [5]:
def sentiment(text):

    # tekst poniżej 510 tokenów
    if len(tokenizer.tokenize(text)) <= 510:

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)

        return probs[0][2].item() - probs[0][0].item()

    # tekst powyżej 510 tokenów
    chunks = split_into_chunks(text, tokenizer)

    scores = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)

        scores.append(
            probs[0][2].item() - probs[0][0].item()
        )

    return sum(scores) / len(scores)

In [3]:
df["token_count"] = df["lyrics"].apply(
    lambda x: len(tokenizer.tokenize(x))
)

100 * (df["token_count"] > 512).mean()

np.float64(31.470588235294116)

In [7]:
tqdm.pandas()
df["sentiment"] = df["lyrics"].progress_apply(sentiment)

100%|██████████| 680/680 [04:32<00:00,  2.49it/s]


In [8]:
df.head()

,year,rank,artist,song,lyrics,token_count,sentiment
0,1950,1,Fats Domino,The Fat Man,"They call, they call me the fat man 'Cause I w...",182,0.591408
1,1950,2,Percy Mayfield,Please Send Me Someone To Love,"Understanding and peace of mind But, if it's n...",281,0.316507
2,1950,3,Ruth Brown,Teardrops From My Eyes,I think of you And that's the time I feel so b...,201,0.455756
3,1950,4,Nat King Cole,Mona Lisa,"Mona Lisa, Mona Lisa, men have named you You'r...",185,-0.310024
4,1950,5,Patti Page,Tennessee Waltz,When an old friend I happened to see I Introdu...,143,-0.554108


In [11]:
df.to_excel("results_roberta.xlsx", index=False)